In [ ]:
!pip install -q \
    "google-auth==2.49.0" \
    "google-genai>=2.12.1,<3" \
    pymupdf \
    faiss-cpu \
    sentence-transformers \
    gradio

In [ ]:
import os
import re
import fitz
import faiss
import numpy as np
import gradio as gr

from google import genai
from google.colab import userdata, files
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [ ]:
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY is missing.")

client = genai.Client(api_key=api_key)

MODEL = "gemini-3.6-flash"

print("Gemini connected!")

Gemini connected!


In [ ]:
response = client.interactions.create(
    model=MODEL,
    input="Say hello. My AI College Assistant is ready."
)

print(response.output_text)

In [ ]:
uploaded = files.upload()

pdf_files = list(uploaded.keys())

knowledge_base = []

for file in pdf_files:

    pdf = fitz.open(file)

    for page_no, page in enumerate(pdf, 1):

        text = page.get_text().strip()

        if text:
            knowledge_base.append({
                "text": text,
                "source": file,
                "page": page_no
            })

    pdf.close()

print("Knowledge base created!")
print("Total pages:", len(knowledge_base))

Saving CollegeAttendance.pdf to CollegeAttendance (2).pdf
Saving CollegeRules.pdf to CollegeRules (2).pdf
Saving college faq.pdf to college faq (1).pdf
Saving College Events.pdf to College Events (1).pdf
Knowledge base created!
Total pages: 18


In [ ]:
def chunk_text(text, size=800, overlap=120):

    chunks = []

    for start in range(0, len(text), size - overlap):

        chunk = text[start:start + size]

        if chunk.strip():
            chunks.append(chunk)

    return chunks
chunks = []

for doc in knowledge_base:

    for text in chunk_text(doc["text"]):

        chunks.append({
            "text": text,
            "source": doc["source"],
            "page": doc["page"]
        })

print("Total chunks:", len(chunks))

Total chunks: 24


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    normalize_embeddings=True
).astype("float32")

print("Embeddings generated:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings generated: (24, 384)


In [ ]:
vector_db = faiss.IndexFlatIP(
    embeddings.shape[1]
)

vector_db.add(embeddings)

print("Vector database created!")
print("Total vectors:", vector_db.ntotal)

Vector database created!
Total vectors: 24


In [ ]:
def retrieve(question):

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = vector_db.search(
        query_embedding,
        min(4, vector_db.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], ids[0]):

        if idx != -1:
            results.append(chunks[idx])

    return results
results = retrieve("What are the attendance rules?")

for item in results:

    print(item["text"][:300])
    print("Source:", item["source"])
    print("Page:", item["page"])
    print("-" * 40)

Attendance Rules & Requirements 
• 
Mandatory Attendance: Students must maintain a minimum of 80% attendance per 
semester to be eligible to appear for semester examinations. [1, 2] 
• 
Condonation: Attendance between 65% and 79% requires a condonation fee. [1] 
• 
Repeating the Semester: Attendance
Source: CollegeAttendance (2).pdf
Page: 1
----------------------------------------
iplinary actions or 
dismissal. [5]  
* Identity Cards: Students must wear and display their college ID cards at all times; misuse or 
tampering with ID cards, bus passes, or library tokens invites punishment. [1, 4]  
 
## Dress Code 
 
* For Male Students: 
* Formal shirts, pants, and shoes are re
Source: CollegeRules (2).pdf
Page: 1
----------------------------------------
Loyola College (primarily referring to Loyola College, Chennai) enforces strict rules and regulations 
regarding attendance, discipline, dress code, and campus conduct. [1]  
## Discipline and Punctuality 
 
* Attendance: High attendance

In [ ]:
SYSTEM_PROMPT = """
You are a Loyola College Chennai AI Assistant.

Answer questions about:
- College rules
- Attendance
- Examinations
- College events

Use only the provided knowledge base.

Do not invent information.

If information is unavailable, say so.

For personal or sensitive questions,
direct students to the appropriate college office.

Always provide source references when available.
"""

In [ ]:
def chatbot(question, history):

    # Retrieve relevant knowledge
    results = retrieve(question)

    if not results:
        return "No relevant information found in the knowledge base."

    # Prepare context
    context = "\n".join(
        item["text"] for item in results
    )

    prompt = f"""
    Question: {question}

    Knowledge Base:
    {context}

    Answer using only the knowledge base.
    """

    try:

        response = client.interactions.create(
            model=MODEL,
            system_instruction=SYSTEM_PROMPT,
            input=prompt
        )

        answer = response.output_text

    except Exception as e:

        return f"Gemini API error: {e}"

    # Add source references
    sources = "\n\nSources:\n"

    for item in results:
        sources += (
            f"- {item['source']}, Page {item['page']}\n"
        )

    return answer + sources

In [ ]:
demo = gr.ChatInterface(
    fn=chatbot,
    title="Loyola College AI Assistant",
    description="Ask questions about college rules, attendance, exams and events."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://594a46786f3381b1bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
